# Arrhythmia Detection with 1D-CNN -- Phase 1 & 2 Baseline Notebook (Kaggle)

End-to-end notebook for the first two research phases at the **100 Hz / 500 Hz scheme**:

1. **Preprocessing** -- light, config-driven cleaning: wavelet db4 (baseline wander removal) + median
   baseline + high-frequency bandpass using the existing bounds (0.5-45 Hz). z-score normalization is
   disabled via `preprocessing.CLEANING_FLAGS` (code kept, stage off). Produces raw ("murni") and
   cleaned tensors at **100 Hz and 500 Hz** for PTB-XL and Chapman. The 250 Hz scheme is deferred.
   *Attach the raw and cleaned datasets as two separate Kaggle datasets; preprocessing auto-skips when
   the cleaned tensors are present.*
2. **Phase 1 (RAW baselines)** -- in-domain, all-class, softmax + native labels, on the **raw** folders.
3. **Phase 2 (CLEANED comparison)** -- identical baselines on the **cleaned** folders, so raw vs
   cleaned can be compared directly.
4. **Post-processing** -- per-experiment confusion matrices, classification reports, misclassified ECG
   plots, a RAW vs CLEANED phase comparison, statistical tests, and a single results archive.

The notebook runs an experiment **just by picking a folder** (`DATA_SELECTION` + `RUN_PHASES` below);
empty folders are skipped automatically. Every run writes an 18-field experiment record
(`experiment_metadata.json`) plus a registry row (`experiment_registry.csv`) under
`output/research_experiments/` -- see `docs/research-progress.md`.

> Use **fast mode** (default) to smoke-test the whole flow, then **full mode** for the reference
> configuration from `docs/training-configs/`.


## How to run on Kaggle

1. Create a Kaggle notebook with **GPU** accelerator (P100 or T4). TensorFlow is preinstalled.
2. Make the repo source available so `src/config` exists in the workspace:
   - either clone/sync the project into `/kaggle/working/arrhythmia-detection-1d-cnn`, or
   - open this notebook inside a copy of the project on your local machine and use *Kaggle -> File ->
     Import -> Notebook*.
3. Provide the data. The raw and cleaned tensors are released as **two separate Kaggle datasets**, each
   mirroring a `dataset/` folder:
   - **Raw dataset**: `dataset/PTBXL/{ptbxl_database.csv, scp_statements.csv, records100, records500}` +
     `dataset/Chapman/{ConditionNames_SNOMED-CT.csv, WFDBRecords}`.
   - **Cleaned dataset**: `dataset/resample/{ptbxl,chapman}_{raw,clean}_{100hz,500hz}/` +
     `dataset/resample/manifest_{ptbxl,chapman}.csv`.
   Attach whatever combination the run needs. The mount cell merges every attached `/kaggle/input/*`
   bundle and auto-detects the layout:
   - **cleaned present** -> preprocessing is **skipped** (fast start);
   - **raw only**       -> preprocessing runs from scratch and also produces the cleaned folders;
   - **native PTB-XL (all classes)** works even on cleaned-only runs because the manifest embeds the
     native SCP label (`native_label`) -- no `ptbxl_database.csv` needed.
4. In the **pick folders** cell set `DATA_SELECTION` to `500Hz` or `100Hz` and choose
   `RUN_PHASES` (Phase 1 raw, Phase 2 cleaned, or both). Phase folders that are empty are skipped, so a
   cleaned-only upload still runs Phase 2 without any code change.
5. Training duration depends on `FAST_MODE` (see the training cells).
6. Download `output/kaggle_all_results.zip` from the last cell.


In [ ]:
import os, sys, subprocess

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

# wfdb + DSP + stats extras (TF / numpy / pandas / sklearn already on Kaggle)
_pip("wfdb", "PyWavelets", "imbalanced-learn", "fastdtw", "neurokit2", "seaborn", "tabulate")

import tensorflow as tf
print("tensorflow", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))


In [ ]:
import os, sys

def _find_repo_root():
    cwd = os.getcwd()
    for cand in [cwd, os.path.join(cwd, "arrhythmia-detection-1d-cnn"), os.path.dirname(cwd)]:
        if os.path.isdir(os.path.join(cand, "src", "config")):
            return os.path.abspath(cand)
    return None

REPO_ROOT = _find_repo_root()
if REPO_ROOT is None:
    raise RuntimeError(
        "Repo source not found. Clone/sync the project into /kaggle/working "
        "(so that src/config exists) before running this notebook."
    )
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("REPO_ROOT =", REPO_ROOT)


In [ ]:
import os, glob, shutil

from src.config import config as cfg
from src.config.experiment_configs import Config

def _has(p):
    return os.path.exists(p)

def _populated(d):
    if not _has(d):
        return False
    try:
        return len([f for f in os.listdir(d) if f.endswith(".npy")]) > 0
    except OSError:
        return False

# ----------------------------------------------------------------------
# 1) Merge every Kaggle input bundle (`/kaggle/input/<slug>/`) that ships a
#    `dataset/` mirror -- supports attaching the RAW and CLEANED datasets
#    separately (Save Version per dataset URL).
# ----------------------------------------------------------------------
mounts = sorted(glob.glob("/kaggle/input/*"))
if not mounts:
    print("No /kaggle/input mounts found - assuming local dataset/ already populated.")

for m in mounts:
    src = m
    if os.path.isdir(os.path.join(m, "dataset")):
        src = os.path.join(m, "dataset")
    found_any = False
    for sub in ["PTBXL", "Chapman", "resample"]:
        p = os.path.join(src, sub)
        if os.path.isdir(p):
            shutil.copytree(p, os.path.join(cfg.DATASET_DIR, sub), dirs_exist_ok=True)
            found_any = True
    if found_any:
        print("Merged input bundle:", src)

# ----------------------------------------------------------------------
# 2) Readiness report (100/500 Hz scheme; raw-only / cleaned-only / both)
# ----------------------------------------------------------------------
FAST_FOLDERS = [
    "ptbxl_raw_100hz", "ptbxl_clean_100hz",
    "ptbxl_raw_500hz", "ptbxl_clean_500hz",
    "chapman_raw_100hz", "chapman_clean_100hz",
    "chapman_raw_500hz", "chapman_clean_500hz",
]

PTB_MAN = os.path.join(cfg.RESAMPLE_BASE, "manifest_ptbxl.csv")
CHAP_MAN = os.path.join(cfg.RESAMPLE_BASE, "manifest_chapman.csv")

PREP_PRESENT = {k: _populated(cfg.SUB_FOLDERS[k]) for k in FAST_FOLDERS}

PTB_PREP = (
    _has(PTB_MAN)
    and any(PREP_PRESENT[k] for k in ["ptbxl_clean_500hz", "ptbxl_clean_100hz",
                                      "ptbxl_raw_500hz", "ptbxl_raw_100hz"])
)
CHAP_PREP = (
    _has(CHAP_MAN)
    and any(PREP_PRESENT[k] for k in ["chapman_clean_500hz", "chapman_raw_500hz",
                                      "chapman_clean_100hz", "chapman_raw_100hz"])
)

PTB_CLEAN_ONLY = _has(PTB_MAN) and any(PREP_PRESENT[k] for k in ["ptbxl_clean_500hz", "ptbxl_clean_100hz"])
CHAP_CLEAN_ONLY = _has(CHAP_MAN) and any(PREP_PRESENT[k] for k in ["chapman_clean_500hz", "chapman_clean_100hz"])

RAW_PTB = _has(cfg.PTBXL_CSV) and len(glob.glob(os.path.join(cfg.DATASET_DIR, "PTBXL", "records*"))) > 0
RAW_CHAP = _has(cfg.CHAPMAN_RECS)

print("--- Readiness ---")
print("PTB-XL preprocessed :", PTB_PREP, "(cleaned-only =", PTB_CLEAN_ONLY, ")")
print("Chapman preprocessed:", CHAP_PREP, "(cleaned-only =", CHAP_CLEAN_ONLY, ")")
print("PTB-XL raw          :", RAW_PTB)
print("Chapman raw         :", RAW_CHAP)

if not PTB_PREP and not RAW_PTB:
    print("WARNING: PTB-XL needs the cleaned dataset (resample/) OR raw dataset/PTBXL attached.")
if not CHAP_PREP and not RAW_CHAP:
    print("WARNING: Chapman needs the cleaned dataset (resample/) OR raw dataset/Chapman attached.")


In [ ]:
import time, runpy

t0 = time.time()

if PTB_PREP:
    print("PTB-XL preprocessed data present - skipping.")
elif RAW_PTB:
    print("Preprocessing PTB-XL (raw + cleaned, 100Hz and 500Hz)...")
    runpy.run_module("src.preprocessing.proccess_ptbxl", run_name="__main__")
    print("PTB-XL preprocessing finished in %.1f min" % ((time.time() - t0) / 60))
else:
    raise RuntimeError("PTB-XL data unavailable (neither preprocessed nor raw).")


In [ ]:
t0 = time.time()

if CHAP_PREP:
    print("Chapman preprocessed data present - skipping.")
elif RAW_CHAP:
    print("Preprocessing Chapman (raw + cleaned, 500Hz native and 100Hz)...")
    runpy.run_module("src.preprocessing.proccess_chapman", run_name="__main__")
    print("Chapman preprocessing finished in %.1f min" % ((time.time() - t0) / 60))
else:
    raise RuntimeError("Chapman data unavailable (neither preprocessed nor raw).")


In [ ]:
# Optional EDA / preprocessing-quality audit (uses the preprocessed folders).
# Both are slow on full data; keep False unless you need the audit figures.
RUN_EDA = False

if RUN_EDA:
    runpy.run_module("src.analysis.eda_visualization", run_name="__main__")
    runpy.run_module("src.analysis.eda_quantitative_audit", run_name="__main__")
    print("EDA finished.")
else:
    print("EDA skipped (set RUN_EDA = True to enable).")


## The Phase 1 & 2 experiment matrix

All baseline runs use the **softmax** head with **native (all-class)** labels, trained and tested
in-domain on the *selected folders*:

| Phase | Folder kind | Experiment IDs |
|-------|-------------|----------------|
| **Phase 1 -- RAW baselines** | raw 100/500 Hz | `RAW_PTBXL_3L`, `RAW_CHAPMAN_3L` |
| **Phase 2 -- CLEANED comparison** | cleaned 100/500 Hz | `CLEAN_PTBXL_3L`, `CLEAN_CHAPMAN_3L` |

Every experiment record pins `dataset_version` (folder + fs), `preprocessing_version`,
`lead_configuration` (I, II, III) and a full environment snapshot. Sigmoid and cross-dataset plans
belong to later phases (gated behind `threshold_tuning` / mapped scheme).

Switching `DATA_SELECTION` from 500 Hz to 100 Hz remaps every folder below, so the **same** phases
are runnable at either sampling rate.


In [ ]:
# Phase 1 & 2 baseline plan builder: softmax + native (all-class) labels.
DATASETS = ["PTBXL", "CHAPMAN"]

PHASE_KINDS = {
    "Phase1_Raw":     {"kind": "raw",   "title": "Phase 1: RAW baselines"},
    "Phase2_Cleaned": {"kind": "clean", "title": "Phase 2: CLEANED baselines"},
}

def _folder(dataset, kind, fs_suffix):
    prefix = "ptbxl" if dataset == "PTBXL" else "chapman"
    return "%s_%s_%s" % (prefix, kind, fs_suffix)

def build_plans(phase_key, folders):
    plans = []
    for ds in DATASETS:
        plans.append({
            "phase": phase_key,
            "folder": folders[ds],
            "scheme": "softmax",
            "label_scheme": "native",
            "train": ds,
            "test": ds,
            "name": "phase_%s_%s_%s" % (phase_key.lower(), ds.lower(), folders[ds]),
        })
    return plans

print("Phase kinds:", {k: v["kind"] for k, v in PHASE_KINDS.items()})


## 1) Pick the folders to run (Phase 1 & 2)

Experiments run **only on the folders you select** here.

| Phase | PTB-XL folder | Chapman folder |
|-------|---------------|----------------|
| Phase 1 (RAW)     | `ptbxl_raw_<fs>`   | `chapman_raw_<fs>`   |
| Phase 2 (CLEANED) | `ptbxl_clean_<fs>` | `chapman_clean_<fs>` |

Set `DATA_SELECTION` to `500Hz` or `100Hz` and keep `RUN_PHASES` (or comment out a phase). Phases
whose folders are empty are **skipped automatically**, so attaching only the cleaned dataset still
runs Phase 2, and only raw runs Phase 1.


In [ ]:
# ----------------------------------------------------------------------
# PHASE FOLDER SELECTION -- just pick the folder here.
# ----------------------------------------------------------------------
DATA_SELECTION = "500Hz"                       # "500Hz" | "100Hz"
RUN_PHASES = ["Phase1_Raw", "Phase2_Cleaned"]  # comment out a phase to skip it

_FS_SUFFIX = {"500Hz": "500hz", "100Hz": "100hz"}[DATA_SELECTION]

for _ph in RUN_PHASES:
    if _ph not in PHASE_KINDS:
        raise ValueError("Unknown phase %r. Options: %s" % (_ph, list(PHASE_KINDS)))

ACTIVE_FOLDERS = {}
for _ph in RUN_PHASES:
    _kind = PHASE_KINDS[_ph]["kind"]
    ACTIVE_FOLDERS[_ph] = {
        ds: _folder(ds, _kind, _FS_SUFFIX) for ds in DATASETS
    }

print("DATA_SELECTION =", DATA_SELECTION)
for _ph, _folders in ACTIVE_FOLDERS.items():
    print(" ", _ph, "->", _folders)
    for _ds, _key in _folders.items():
        _ok = _populated(cfg.SUB_FOLDERS[_key])
        print("     %-7s %-24s %s" % (_ds, _key, "OK" if _ok else "EMPTY"))

_empty = []
for _ph, _folders in ACTIVE_FOLDERS.items():
    if not any(_populated(cfg.SUB_FOLDERS[k]) for k in _folders.values()):
        _empty.append(_ph)
if _empty:
    print("WARNING: empty phases (will be skipped in training):", _empty)


In [ ]:
# FAST_MODE renders every phase quickly (few epochs, single architecture) so
# the notebook can run end-to-end on Kaggle. Set FAST_MODE = False for the
# full reference configuration (docs/training-configs).
FAST_MODE = True

if FAST_MODE:
    Config.EPOCHS = 5
    Config.FILTER_SPACES = {"Small": [32, 64, 128, 128, 256]}
    Config.KERNEL_SPACES = {"Balanced": [15, 11, 7, 5, 3]}
    Config.DILATION_SPACES = {"Progressive_Dilation": [1, 2, 4, 8, 16]}
    Config.TEMPORAL_MODELS = ["Pure_CNN"]
    Config.UNDERSAMPLE_RATIO = None
    Config.OVERSAMPLE_METHOD = None
else:
    Config.EPOCHS = 35
    Config.FILTER_SPACES = {"Medium": [64, 128, 256, 256, 512]}
    Config.KERNEL_SPACES = {"Balanced": [15, 11, 7, 5, 3]}
    Config.DILATION_SPACES = {"Progressive_Dilation": [1, 2, 4, 8, 16]}
    Config.TEMPORAL_MODELS = ["Pure_CNN"]
    Config.UNDERSAMPLE_RATIO = 10
    Config.OVERSAMPLE_METHOD = "smote_tomek"
    Config.OVERSAMPLE_STRATEGY = {"AF": 3000, "Bradikardia": 3000, "Takikardia": 3000}

print("FAST_MODE =", FAST_MODE, "| epochs =", Config.EPOCHS,
      "| selection =", DATA_SELECTION,
      "| phases =", RUN_PHASES)


## Training: Phase 1 & 2 baselines

Each selected phase sets the active folders on `Config`, then every baseline plan drives the unified
runner (`src/experiments/run_experiment.py`) in-process. Outputs per experiment (config JSON, training
log, best model, confusion matrix, classification report, misclassified plots, plus the 18-field
`experiment_metadata.json` record) are saved by the runner under `output/research_experiments/`.


In [ ]:
import runpy, gc
import pandas as pd
import tensorflow as tf

RESULT_ROOT = os.path.join(cfg.OUTPUT_DIR, "research_experiments")
TRACKER = os.path.join(RESULT_ROOT, Config.MASTER_TRACKER_CSV)
PLAN_SUMMARY = os.path.join(RESULT_ROOT, "plan_summary.csv")

def _read_tracker():
    if not os.path.exists(TRACKER):
        return pd.DataFrame()
    return pd.read_csv(TRACKER)

def _write_plan_log(rows):
    pd.DataFrame(rows).to_csv(PLAN_SUMMARY, index=False)

def _set_plan_config(plan):
    Config.SCHEME = plan["scheme"]
    Config.LABEL_SCHEME = plan["label_scheme"]
    Config.TRAIN_DATASET = plan["train"]
    Config.TEST_DATASET = plan["test"]

base = _read_tracker()
known = set(base["Experiment"].astype(str)) if len(base) else set()
plan_rows = []

for ph in RUN_PHASES:
    folders = ACTIVE_FOLDERS[ph]
    print("\n" + "=" * 70)
    print("%s | folders=%s" % (PHASE_KINDS[ph]["title"], folders))
    print("=" * 70)

    if not any(_populated(cfg.SUB_FOLDERS[k]) for k in folders.values()):
        print("SKIP: no data in phase folders ->", folders)
        continue

    Config.FOLDER_PTBXL = folders["PTBXL"]
    Config.FOLDER_CHAPMAN = folders["CHAPMAN"]

    for plan in build_plans(ph, folders):
        print("\n  PLAN: %s  |  %s / %s  |  train=%s test=%s  |  folder=%s"
              % (plan["name"], plan["scheme"], plan["label_scheme"],
                 plan["train"], plan["test"], plan["folder"]))

        _set_plan_config(plan)

        row = dict(plan)
        try:
            runpy.run_module("src.experiments.run_experiment", run_name="__main__")
            tf.keras.backend.clear_session()
            gc.collect()

            now = _read_tracker()
            new = now[~now["Experiment"].astype(str).isin(known)] if len(now) else now
            known = set(now["Experiment"].astype(str)) if len(now) else set()

            if len(new):
                top = new.sort_values("Macro_F1", ascending=False).iloc[0]
                row.update({
                    "runs": int(len(new)),
                    "best_experiment": str(top["Experiment"]),
                    "best_Macro_F1": float(top["Macro_F1"]),
                    "best_Balanced_Accuracy": float(top["Balanced_Accuracy"]),
                    "error": "",
                })
            else:
                row.update({"runs": 0, "best_experiment": "", "best_Macro_F1": float("nan"),
                            "best_Balanced_Accuracy": float("nan"), "error": "no rows"})
        except Exception as e:
            row.update({"runs": 0, "best_experiment": "", "best_Macro_F1": float("nan"),
                        "best_Balanced_Accuracy": float("nan"), "error": str(e)})
            print("  PLAN FAILED:", plan["name"], "->", e)

        plan_rows.append(row)
        _write_plan_log(plan_rows)
        print("  Recorded:", row["name"], "| runs =", row["runs"],
              "| best F1 =", row["best_Macro_F1"])

print("\nAll phases finished. Per-plan log:", PLAN_SUMMARY)


## Post-processing: RAW vs CLEANED phase comparison

Phase 1 and Phase 2 results are compared per dataset and sampling rate: best Macro-F1 and balanced
accuracy on the held-out test split of each baseline. The comparison table and figure are written under
`output/research_experiments/`.


In [ ]:
import os
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PHASE_COMP = os.path.join(RESULT_ROOT, "phase_comparison.csv")

if not os.path.exists(PLAN_SUMMARY):
    print("plan_summary.csv not found - run the training cell first.")
else:
    log = pd.read_csv(PLAN_SUMMARY)
    log["error"] = log["error"].fillna("").astype(str)
    log = log[log["error"] == ""]

    best = log.sort_values("best_Macro_F1", ascending=False) \
              .drop_duplicates(subset=["phase", "train", "folder"], keep="first")

    rows = []
    for ds in DATASETS:
        sub = best[best["train"] == ds]
        for fs_sfx in ["500hz", "100hz"]:
            raw = sub[(sub["folder"] == _folder(ds, "raw", fs_sfx)) & (sub["phase"] == "Phase1_Raw")]
            cln = sub[(sub["folder"] == _folder(ds, "clean", fs_sfx)) & (sub["phase"] == "Phase2_Cleaned")]
            row = {"dataset": ds, "fs": fs_sfx}
            if len(raw):
                row.update({"RAW_Macro_F1": float(raw["best_Macro_F1"].iloc[0]),
                            "RAW_Balanced_Accuracy": float(raw["best_Balanced_Accuracy"].iloc[0]),
                            "RAW_experiment": str(raw["best_experiment"].iloc[0])})
            if len(cln):
                row.update({"CLEAN_Macro_F1": float(cln["best_Macro_F1"].iloc[0]),
                            "CLEAN_Balanced_Accuracy": float(cln["best_Balanced_Accuracy"].iloc[0]),
                            "CLEAN_experiment": str(cln["best_experiment"].iloc[0])})
            if "RAW_Macro_F1" in row and "CLEAN_Macro_F1" in row:
                row["Delta_Macro_F1"] = row["CLEAN_Macro_F1"] - row["RAW_Macro_F1"]
            rows.append(row)

    comp = pd.DataFrame(rows)
    comp.to_csv(PHASE_COMP, index=False)
    print("\nPhase comparison saved:", PHASE_COMP)
    print(comp.to_string(index=False))

    # Grouped bar: CLEAN vs RAW Macro-F1 per dataset x fs
    if len(comp):
        long = comp.melt(id_vars=["dataset", "fs"],
                         value_vars=["RAW_Macro_F1", "CLEAN_Macro_F1"],
                         var_name="Phase", value_name="Macro_F1")
        long["dataset"] = long["dataset"] + " " + long["fs"].str.upper()
        plt.figure(figsize=(11, 6))
        ax = sns.barplot(data=long, x="dataset", y="Macro_F1", hue="Phase")
        plt.title("Raw vs Cleaned Baseline Macro-F1", fontsize=14, fontweight="bold")
        plt.xticks(rotation=15)
        for p in ax.patches:
            h = p.get_height()
            if not np.isnan(h) and h > 0:
                ax.annotate("%.3f" % h, (p.get_x() + p.get_width() / 2, h),
                            ha="center", va="bottom", fontsize=8)
        plt.tight_layout()
        plt.savefig(os.path.join(RESULT_ROOT, "phase_comparison.png"), dpi=150)
        plt.close()

    # Standardized phase_comparison.csv for the results cell + zip
    print("done")


In [ ]:
# Statistical significance analysis over the unified master tracker.
runpy.run_module("src.evaluation.run_statistical_tests", run_name="__main__")


## Results summary & download

Everything recorded during this run is shown below and packaged into a single
zip archive for download.


In [ ]:
import os, zipfile
import pandas as pd
from IPython.display import display, Markdown

RESULT_ROOT = os.path.join(cfg.OUTPUT_DIR, "research_experiments")

trk = pd.read_csv(os.path.join(RESULT_ROOT, Config.MASTER_TRACKER_CSV))
display(Markdown("### Master experiment tracker"))
display(trk.sort_values("Macro_F1", ascending=False).reset_index(drop=True))

ps = os.path.join(RESULT_ROOT, "plan_summary.csv")
if os.path.exists(ps):
    display(Markdown("### Per-plan summary (Phase 1 & 2)"))
    display(pd.read_csv(ps))

pc = os.path.join(RESULT_ROOT, "phase_comparison.csv")
if os.path.exists(pc):
    display(Markdown("### RAW vs CLEANED phase comparison"))
    display(pd.read_csv(pc))

# ----------------------------------------------------------------------
# Package everything for download
# ----------------------------------------------------------------------
ZIP_NAME = os.path.join(cfg.OUTPUT_DIR, "kaggle_all_results.zip")
with zipfile.ZipFile(ZIP_NAME, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in ["research_experiments", "statistical_tests", "cross_dataset_test"]:
        root = os.path.join(cfg.OUTPUT_DIR, folder)
        if os.path.isdir(root):
            for dp, _, files in os.walk(root):
                for fn in files:
                    p = os.path.join(dp, fn)
                    zf.write(p, os.path.relpath(p, cfg.OUTPUT_DIR))
print("\nAll results zipped to:", ZIP_NAME)


## Where to find the outputs

- **Master tracker**: `output/research_experiments/master_experiment_tracker.csv`
- **Per-experiment folders**: `output/research_experiments/<scheme>_<label_scheme>/<experiment>/`
  (config JSON, `training_log.csv`, `best_model.keras`, `metrics.csv`,
  `confusion_matrix.png`, `classification_report.txt`, `misclassified/` plots, `experiment_metadata.json`)
- **Experiment registry**: `output/research_experiments/experiment_registry.csv`
- **Phase comparison (RAW vs CLEANED)**: `output/research_experiments/phase_comparison.csv` + `phase_comparison.png`
- **Statistical tests**: `output/statistical_tests/*.csv`
- **Everything zipped**: `output/kaggle_all_results.zip`

Switch `FAST_MODE = False` (training cell) and re-run cells 9+ for the full reference configuration
from `docs/training-configs/`.
